# Demo 06a — Decision Trees

**Class 06 · Blocks 1-4** (slides: `slides/06_trees_knn.md`)

Eight live demos, each with more experiments than the slides show:

* **B1 — Models without a formula:** the regions of a linear model, a tree, 1-NN and k-NN on the same points, as sets of cells; how fast each model approaches a circular target; the price in dimension.
* **T1 — Impurity and the best split:** entropy and Gini by hand on a 14-row table, then on real data.
* **T2 — A tree from scratch:** the greedy algorithm in 40 lines of NumPy, against scikit-learn.
* **T3 — Depth, overfitting and pruning:** validation curves, cost-complexity pruning, the rules of a small tree.
* **T4 — Regression trees:** variance reduction, the staircase, the failure outside the data.
* **T5 — Instability and importances:** many trees from many samples, and why importances can lie.
* **T6 — Feature types and encoding:** the type of each column, five encodings of a category, the number of questions on a category, distances between categories, many levels and missing values.
* **T7 — Counting the trees:** all trees of depth 2 against the greedy tree; the cost of growth that makes the best tree NP-complete.

In [ ]:
import math
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display
from matplotlib.patches import Rectangle
from scipy.spatial import Delaunay, Voronoi, voronoi_plot_2d
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_moons
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import (
    RepeatedKFold,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 4)
pl.Config.set_tbl_rows(40)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
DATA = Path("../../datasets")

sys.path.append("..")
import plotlib  # noqa: E402

plotlib.setup(module="06")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line: the frame title of the deck."""
    print(f"--- slide: {title} ---")


def show(frame: pl.DataFrame, digits: int = 3) -> None:
    """Print a polars table with rounded floats."""
    print(frame.with_columns(pl.col(pl.Float64).round(digits)))




def table(rows, cols):
    """Print a list of rows as a polars table (hyperparameters are written with `:g`)."""
    keep = {"alpha", "ccp_alpha", "max_depth", "min_leaf", "k"}
    rows = [tuple(f"{v:g}" if c in keep and isinstance(v, float) else v for c, v in zip(cols, r)) for r in rows]
    print(pl.DataFrame(rows, schema=cols, orient="row"))


def load_table(name, target):
    """Read `datasets/<name>.csv.zst` with polars: null -> median / "missing", text -> one-hot."""
    df = pl.read_csv(DATA / f"{name}.csv.zst")
    X = df.drop(target)
    X = X.with_columns([pl.col(c).fill_null(pl.col(c).median()) for c, t in X.schema.items() if t.is_numeric()])
    cats = [c for c, t in X.schema.items() if t == pl.String]
    if cats:
        X = X.with_columns([pl.col(c).fill_null("missing") for c in cats]).to_dummies(columns=cats)
    return X.to_numpy().astype(float), df[target].to_numpy(), X.columns


def load_california():
    """California housing with the logarithm of the four skewed features (Class 03)."""
    X, y, names = load_table("california_housing", "MedHouseVal")
    for j, n in enumerate(names):
        if n in ("AveRooms", "AveBedrms", "Population", "AveOccup"):
            X[:, j] = np.log(X[:, j])
    return X, y.astype(float), names


Xb, yb, names_b = load_table("breast_cancer", "target")
Xg, cls_g, names_g = load_table("german_credit", "class")
yg = (cls_g == "bad").astype(int)
Xh, num_h, names_h = load_table("heart_disease", "num")
yh = (num_h == ">50_1").astype(int)
Xw, q_w, names_w = load_table("wine_quality", "quality")
yw = (q_w >= 6).astype(int)
Xd, yd, names_d = load_table("digits", "target")
CLASSIFICATION = {"breast_cancer": (Xb, yb), "german_credit": (Xg, yg), "heart_disease": (Xh, yh), "wine (quality >= 6)": (Xw, yw), "digits": (Xd, yd.astype(int))}
for k, (X_, y_) in CLASSIFICATION.items():
    print(f"{k:<22} {X_.shape[0]:>5} rows x {X_.shape[1]:>3} features, classes {np.bincount(y_).tolist()}")

## B1 — Models without a formula: regions of the feature space

A linear model, a tree and k-NN all end in the same object: a **partition of the feature space $\mathcal{X}$ into cells**,
each cell with one label. What differs is the *shape* of the cells: a half-plane, a box, a Voronoi polygon. Here the same
50 points (two moons, $\sigma = 0.22$) go through every model and the regions are drawn, not sketched.

In [ ]:
# slide: Linear Model and Tree: Half-Planes and Boxes
slide("Linear Model and Tree: Half-Planes and Boxes")
Xr, yr = make_moons(n_samples=50, noise=0.22, random_state=6)
XLIM, YLIM = (-1.6, 2.6), (-1.0, 1.6)
gx, gy = np.meshgrid(np.linspace(*XLIM, 420), np.linspace(*YLIM, 260))
GRID = np.c_[gx.ravel(), gy.ravel()]
CLASS_COLOURS = [BLUE, ORANGE]


def regions(ax, model, title):
    """Fill the region of each class with the model's prediction on a fine grid, then draw the 50 points."""
    z = model.predict(GRID).reshape(gx.shape)
    ax.contourf(gx, gy, z, levels=[-0.5, 0.5, 1.5], colors=CLASS_COLOURS, alpha=0.22)
    for c in (0, 1):
        ax.scatter(*Xr[yr == c].T, s=7, color=CLASS_COLOURS[c], edgecolor="white", linewidth=0.3, zorder=3)
    frame(ax, title)


def frame(ax, title):
    """The feature space as a thin rectangle: no ticks, no grid, a two-line title."""
    ax.set(xlim=XLIM, ylim=YLIM, xticks=[], yticks=[])
    ax.set_title(title, fontsize=7.5, pad=3)
    ax.grid(False)
    ax.set_aspect("equal")
    for spine in ax.spines.values():
        spine.set(visible=True, color="#999999", linewidth=0.6)


def leaf_boxes(tree):
    """The leaves of a fitted tree as boxes (x0, y0, x1, y1) with their class: the cells of the partition."""
    t = tree.tree_
    boxes = []

    def walk(node, box):
        if t.children_left[node] == -1:
            boxes.append((box, int(np.argmax(t.value[node]))))
            return
        j, thr = int(t.feature[node]), float(t.threshold[node])
        left, right = list(box), list(box)
        left[2 + j], right[j] = thr, thr
        walk(t.children_left[node], left)
        walk(t.children_right[node], right)

    walk(0, [XLIM[0], YLIM[0], XLIM[1], YLIM[1]])
    return boxes


linear = LogisticRegression().fit(Xr, yr)
tree3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xr, yr)
tree8 = DecisionTreeClassifier(max_depth=8, random_state=0).fit(Xr, yr)
rows = []
for name, model in (("linear (logistic)", linear), ("tree, depth 3", tree3), ("tree, depth 8", tree8)):
    cells = 2 if name.startswith("linear") else len(leaf_boxes(model))
    rows.append({"model": name, "cells": cells, "train accuracy": float(model.score(Xr, yr))})
show(pl.DataFrame(rows), 3)
w = linear.coef_[0]
print(
    f"linear model: the border is the line {w[0]:.2f} x1 + {w[1]:.2f} x2 + {linear.intercept_[0]:.2f} = 0: one cut, two half-planes"
)

In [ ]:
# plot b1_models
fg, axs = plotlib.fig("full", 1.35, ncols=3)
regions(axs[0], linear, f"linear\n{rows[0]['cells']} half-planes")
for ax, model, row in zip(axs[1:], (tree3, tree8), rows[1:]):
    regions(ax, model, f"tree, depth {model.get_depth()}\n{row['cells']} boxes")
    for (x0, y0, x1, y1), _ in leaf_boxes(model):
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="#444444", linewidth=0.5, zorder=2))
display(fg)
plotlib.save(fg, "b1_models")

In [ ]:
# slide: Nearest Neighbours: Voronoi Cells
slide("Nearest Neighbours: Voronoi Cells")
vor = Voronoi(Xr)
tri = Delaunay(Xr)
knn1 = KNeighborsClassifier(1).fit(Xr, yr)
knn15 = KNeighborsClassifier(15).fit(Xr, yr)
border = [(a, b) for a, b in vor.ridge_points if yr[a] != yr[b]]
print(
    f"1-NN: {len(Xr)} points give {len(Xr)} Voronoi cells; {len(vor.ridge_points)} cells touch another cell, {len(border)} of those borders separate two classes"
)
print(
    f"Delaunay triangulation of the same points: {len(tri.simplices)} triangles (it joins every pair of cells that share a border)"
)
print(f"train accuracy: 1-NN {knn1.score(Xr, yr):.3f}, 15-NN {knn15.score(Xr, yr):.3f}")
# the cells of 15-NN: how many distinct sets of 15 neighbours the grid sees
nb15 = knn15.kneighbors(GRID, return_distance=False)
cells15 = len({tuple(sorted(r)) for r in nb15})
print(
    f"15-NN: the grid meets {cells15} distinct neighbour sets: the cells are the intersections of many Voronoi-like regions"
)

In [ ]:
# plot b1_knn
fg, axs = plotlib.fig("full", 1.35, ncols=3)
regions(axs[0], knn1, "1-NN\n50 Voronoi cells")
voronoi_plot_2d(
    vor, ax=axs[0], show_vertices=False, show_points=False, line_colors="#444444", line_width=0.5, line_alpha=1
)
axs[0].set(xlim=XLIM, ylim=YLIM)
regions(axs[1], knn1, "Delaunay triangles\nwho is next to whom")
axs[1].triplot(Xr[:, 0], Xr[:, 1], tri.simplices, color="#444444", linewidth=0.45, zorder=2)
regions(axs[2], knn15, "15-NN\nsmooth border")
display(fg)
plotlib.save(fg, "b1_knn")

In [ ]:
# slide: A Class Is a Set of Cells
slide("A Class Is a Set of Cells")
cell_class = knn1.predict(GRID)
share = {c: float(np.mean(cell_class == c)) for c in (0, 1)}
n_cells = {c: int(np.sum(yr == c)) for c in (0, 1)}
overlap = float(np.mean((cell_class == 0) & (cell_class == 1)))
print(
    f"A_0: union of {n_cells[0]} cells, {share[0]:.3f} of the box; A_1: union of {n_cells[1]} cells, {share[1]:.3f} of the box"
)
print(
    f"overlap of A_0 and A_1: {overlap:.3f}; union of A_0 and A_1: {share[0] + share[1]:.3f} of the box (a partition: disjoint and complete)"
)

In [ ]:
# plot b1_sets
fg, axs = plotlib.fig("full", 1.35, ncols=3)
regions(axs[0], knn1, f"$\\mathcal{{X}}$\n{len(Xr)} cells")
voronoi_plot_2d(
    vor, ax=axs[0], show_vertices=False, show_points=False, line_colors="#444444", line_width=0.5, line_alpha=1
)
for ax, c, label in ((axs[1], 1, "$A_1$"), (axs[2], 0, "$A_0$")):
    z = (cell_class == c).reshape(gx.shape).astype(float)
    ax.contourf(gx, gy, z, levels=[0.5, 1.5], colors=[CLASS_COLOURS[c]], alpha=0.35)
    ax.contour(gx, gy, z, levels=[0.5], colors=["#444444"], linewidths=0.7)
    ax.scatter(*Xr[yr == c].T, s=7, color=CLASS_COLOURS[c], edgecolor="white", linewidth=0.3, zorder=3)
    frame(ax, f"{label}\nunion of {n_cells[c]} cells")
axs[0].set(xlim=XLIM, ylim=YLIM)
display(fg)
plotlib.save(fg, "b1_sets")

In [ ]:
# slide: Any Region, Given Enough Cells
slide("Any Region, Given Enough Cells")


def median_radius(d):
    """Radius of the ball around the origin that holds half of the cube [-1, 1]^d (Monte Carlo, fixed seed)."""
    return float(np.median(np.linalg.norm(np.random.default_rng(0).uniform(-1, 1, (200_000, d)), axis=1)))


RADIUS = {d: median_radius(d) for d in (2, 4, 8, 12, 16, 24)}


def ball_data(n, d, rng, noise=0.10):
    """Uniform points of [-1, 1]^d; class 1 inside the ball that holds half of the cube (the disk for d = 2); `noise` of the labels flipped."""
    X = rng.uniform(-1, 1, size=(n, d))
    y = (np.linalg.norm(X, axis=1) < RADIUS[d]).astype(int)
    return X, np.where(rng.random(n) < noise, 1 - y, y)


def odd(k):
    return int(k) | 1


def ball_errors(n, d, seed, n_test=8000):
    """Test error of four models trained on n noisy points of the ball problem (test labels are noisy too: the best possible is 0.10)."""
    rng = np.random.default_rng(seed)
    X, y = ball_data(n, d, rng)
    Xt, yt = ball_data(n_test, d, rng)
    leaf = max(1, int(np.sqrt(n)))
    k = odd(np.sqrt(n))
    algo = "brute" if d > 8 else "auto"
    models = {
        "linear": LogisticRegression(),
        "tree": DecisionTreeClassifier(min_samples_leaf=leaf, random_state=seed),
        "1-NN": KNeighborsClassifier(1, algorithm=algo),
        "k-NN": KNeighborsClassifier(k, algorithm=algo),
    }
    res = {name: float(np.mean(m.fit(X, y).predict(Xt) != yt)) for name, m in models.items()}
    res["leaves"] = int(models["tree"].get_n_leaves())
    return res


NS = [25, 50, 100, 200, 400, 800, 1600, 3200, 6400]
REPEATS = 8
rows = []
for n in NS:
    runs = [ball_errors(n, 2, seed) for seed in range(REPEATS)]
    row = {"n": n, "k": odd(np.sqrt(n)), "leaf size": max(1, int(np.sqrt(n)))}
    for name in ("linear", "tree", "1-NN", "k-NN"):
        row[name] = float(np.mean([r[name] for r in runs]))
        row[name + " sd"] = float(np.std([r[name] for r in runs]))
    row["tree leaves"] = float(np.mean([r["leaves"] for r in runs]))
    rows.append(row)
approx = pl.DataFrame(rows)
show(approx.select("n", "k", "leaf size", "tree leaves", "linear", "tree", "1-NN", "k-NN"), 3)
print("label noise 0.10: Bayes error 0.100; Cover-Hart limit of 1-NN 2 x 0.1 x 0.9 = 0.180")

In [ ]:
# plot b1_approx
fg, ax = plotlib.fig("full", 1.0)
for name, colour, marker in (("linear", GREY, "o"), ("1-NN", PURPLE, "D"), ("tree", GREEN, "s"), ("k-NN", BLUE, "^")):
    m, s = approx[name].to_numpy(), approx[name + " sd"].to_numpy()
    ax.plot(approx["n"], m, color=colour, marker=marker, ms=3, label=name)
    ax.fill_between(approx["n"], m - s, m + s, color=colour, alpha=0.15, linewidth=0)
ax.axhline(0.10, color=RED, ls="--", lw=0.9, label="Bayes error 0.10")
ax.axhline(0.18, color=PURPLE, ls=":", lw=0.9, label="1-NN limit 0.18")
ax.set(xscale="log", xlabel="training examples $n$ (log scale)", ylabel="test error", ylim=(0.05, 0.62))
ax.legend(ncols=3, loc="upper center", bbox_to_anchor=(0.5, 1.32), fontsize=7)
display(fg)
plotlib.save(fg, "b1_approx")

In [ ]:
# slide: The Price: Data and Dimension
slide("The Price: Data and Dimension")
DIMS = [2, 4, 8, 12, 16, 24]
rows = []
for n in (500, 5000):
    for d in DIMS:
        runs = [ball_errors(n, d, seed, n_test=3000) for seed in range(4)]
        rows.append(
            {
                "n": n,
                "d": d,
                "linear": float(np.mean([r["linear"] for r in runs])),
                "tree": float(np.mean([r["tree"] for r in runs])),
                "k-NN": float(np.mean([r["k-NN"] for r in runs])),
                "cells of side 0.5": 4.0**d,
            }
        )
price = pl.DataFrame(rows)
print(price.with_columns(pl.col("cells of side 0.5").map_elements(lambda v: f"{v:.0e}", return_dtype=pl.String)))
print(
    "cells of side 0.5 in the cube of side 2: 4^d, so 16 cells in 2-D and about 17 million in 12-D: the number of cells grows exponentially, the n = 5000 points do not"
)

In [ ]:
# plot b1_price
fg, ax = plotlib.fig("full", 1.1)
for n, ls in ((500, "--"), (5000, "-")):
    sub = price.filter(pl.col("n") == n)
    ax.plot(sub["d"], sub["k-NN"], color=BLUE, ls=ls, marker="^", ms=3, label=f"k-NN, $n$ = {n}")
    ax.plot(sub["d"], sub["tree"], color=GREEN, ls=ls, marker="s", ms=3, label=f"tree, $n$ = {n}")
ax.axhline(0.10, color=RED, ls="--", lw=0.9, label="Bayes error 0.10")
ax.set(
    xscale="log", xlabel="dimension $d$, all features informative (log scale)", ylabel="test error", ylim=(0.05, 0.55)
)
ax.set_xticks(DIMS, [str(d) for d in DIMS])
ax.minorticks_off()
ax.legend(ncols=3, loc="upper center", bbox_to_anchor=(0.5, 1.32), fontsize=7)
display(fg)
plotlib.save(fg, "b1_price")

## T1 — Impurity and the best split

A tree asks a sequence of yes/no questions. Each question should make the groups it creates **purer**: more
examples of one class. The **impurity** of a group with class proportions $p_1,\dots,p_K$ is

* **entropy** $H = -\sum_k p_k \log_2 p_k$ (the bits needed to name the class, Class 01),
* **Gini** $G = 1 - \sum_k p_k^2$ (the probability that two random examples differ),
* **misclassification error** $E = 1 - \max_k p_k$.

The **gain** of a question is the impurity of the parent minus the weighted impurity of the children.

Start with the textbook example: 14 days, "play tennis?" (9 yes, 5 no).

In [ ]:
tennis = pl.DataFrame({
    "outlook": ["sunny", "sunny", "overcast", "rain", "rain", "rain", "overcast", "sunny", "sunny", "rain", "sunny", "overcast", "overcast", "rain"],
    "temperature": ["hot", "hot", "hot", "mild", "cool", "cool", "cool", "mild", "cool", "mild", "mild", "mild", "hot", "mild"],
    "humidity": ["high", "high", "high", "high", "normal", "normal", "normal", "high", "normal", "normal", "normal", "high", "normal", "high"],
    "windy": ["no", "yes", "no", "no", "no", "yes", "yes", "no", "no", "no", "yes", "yes", "no", "yes"],
    "play": ["no", "no", "yes", "yes", "yes", "no", "yes", "no", "yes", "yes", "yes", "yes", "yes", "no"],
})


def entropy(counts):
    counts = np.asarray(counts, dtype=float)
    p = counts / counts.sum(axis=-1, keepdims=True)
    return -np.sum(np.where(p > 0, p * np.log2(np.where(p > 0, p, 1)), 0.0), axis=-1)


def gini(counts):
    counts = np.asarray(counts, dtype=float)
    p = counts / counts.sum(axis=-1, keepdims=True)
    return 1.0 - np.sum(p**2, axis=-1)


def error(counts):
    counts = np.asarray(counts, dtype=float)
    return 1.0 - counts.max(axis=-1) / counts.sum(axis=-1)


yes = (tennis["play"] == "yes").to_numpy()
parent = np.array([(~yes).sum(), yes.sum()])
print(f"parent: {parent[1]} yes, {parent[0]} no -> entropy {entropy(parent):.3f}, Gini {gini(parent):.3f}, error {error(parent):.3f}")
rows = []
for feat in ("outlook", "temperature", "humidity", "windy"):
    res = {"entropy": entropy(parent), "gini": gini(parent), "error": error(parent)}
    for crit, fn in (("entropy", entropy), ("gini", gini), ("error", error)):
        child = 0.0
        for v in tennis[feat].unique().to_list():
            m = (tennis[feat] == v).to_numpy()
            c = np.array([(~yes[m]).sum(), yes[m].sum()])
            child += m.mean() * fn(c)
        res[crit] -= child
    rows.append((feat, res["entropy"], res["gini"], res["error"]))
table(rows, ["question: split on", "gain (entropy)", "gain (Gini)", "gain (error)"])

In [ ]:
# slide: Which Question First?
slide("Which Question First?")
gains_tbl = rows  # the gains of the four tennis questions (cell above), kept for the plot
quest = {"A": ((30, 10), (10, 30)), "B": ((40, 10), (0, 30))}  # (yes, no) of the left and the right group
parent80 = np.array([40, 40])
gain_q = {}
for q_, (left, right) in quest.items():
    left, right = np.array(left), np.array(right)
    gain_q[q_] = float(gini(parent80) - left.sum() / 80 * gini(left) - right.sum() / 80 * gini(right))
    print(
        f"question {q_}: left {left.tolist()}, right {right.tolist()}; Gini of the children "
        f"{gini(left):.3f} and {gini(right):.3f}; gain {gain_q[q_]:.3f}"
    )
print(f"parent (40, 40): Gini {gini(parent80):.3f}")

In [ ]:
# plot t1_questions
fg, axs = plotlib.fig("full", 1.3, ncols=2, sharey=True)
for ax, (q_, (left, right)) in zip(axs, quest.items()):
    for pos, (n_yes, n_no) in enumerate((left, right)):
        ax.bar(pos, n_yes, color=GREEN, width=0.55)
        ax.bar(pos, n_no, bottom=n_yes, color=RED, width=0.55)
        if n_yes:
            ax.text(pos, n_yes / 2, f"{n_yes} yes", ha="center", va="center", fontsize=7.5, color="white")
        if n_no:
            ax.text(pos, n_yes + n_no / 2, f"{n_no} no", ha="center", va="center", fontsize=7.5, color="white")
    ax.set_xticks([0, 1], ["left group", "right group"])
    ax.set_title(f"question {q_}: gain {gain_q[q_]:.3f}", fontsize=7.5, pad=3)
axs[0].set_ylabel("examples")
display(fg)
plotlib.save(fg, "t1_questions")

In [ ]:
# slide: Worked Example: Play Tennis? (1/2)
slide("Worked Example: Play Tennis? (1/2)")
yes = (tennis["play"] == "yes").to_numpy()
parent = np.array([(~yes).sum(), yes.sum()])
print(f"parent: {parent[1]} yes, {parent[0]} no -> entropy {entropy(parent):.3f}")
groups = []
for feat in ("outlook", "humidity"):
    for v in tennis[feat].unique(maintain_order=True).to_list():
        m = (tennis[feat] == v).to_numpy()
        c = np.array([(~yes[m]).sum(), yes[m].sum()])
        groups.append({"question": feat, "value": v, "yes": int(c[1]), "no": int(c[0]), "entropy": float(entropy(c))})
tennis_groups = pl.DataFrame(groups)
show(tennis_groups)

In [ ]:
# plot t1_tennis
fg, ax = plotlib.fig("full", 1.5)
labels_ = [f"{r['question']}: {r['value']}" for r in groups]
ypos = np.arange(len(groups))[::-1]
for y_, r in zip(ypos, groups):
    ax.barh(y_, r["yes"], color=GREEN, height=0.6)
    ax.barh(y_, r["no"], left=r["yes"], color=RED, height=0.6)
    ax.text(r["yes"] + r["no"] + 0.2, y_, f"entropy {abs(r['entropy']):.3f}", va="center", fontsize=7.5)
    ax.text(r["yes"] / 2, y_, str(r["yes"]) if r["yes"] else "", ha="center", va="center", fontsize=7, color="white")
    ax.text(
        r["yes"] + r["no"] / 2, y_, str(r["no"]) if r["no"] else "", ha="center", va="center", fontsize=7, color="white"
    )
ax.set_yticks(ypos, labels_)
ax.set(xlabel="days (green: play, red: no play)", xlim=(0, 11))
ax.grid(False)
display(fg)
plotlib.save(fg, "t1_tennis")

In [ ]:
# slide: Worked Example: Play Tennis? (2/2)
slide("Worked Example: Play Tennis? (2/2)")
outlook = tennis_groups.filter(pl.col("question") == "outlook")
weighted = float(sum((r["yes"] + r["no"]) / 14 * r["entropy"] for r in outlook.to_dicts()))
print(f"outlook: weighted entropy after the question {weighted:.3f}; gain {entropy(parent) - weighted:.3f}")
show(pl.DataFrame(gains_tbl, schema=["question", "gain (entropy)", "gain (Gini)", "gain (error)"], orient="row"))

In [ ]:
# plot t1_gains
fg, ax = plotlib.fig("full", 1.4)
width = 0.26
for j, (crit, colour) in enumerate((("entropy", BLUE), ("Gini", ORANGE), ("error", GREY))):
    vals = [r[1 + j] for r in gains_tbl]
    ax.bar(np.arange(4) + (j - 1) * width, vals, width, color=colour, label=crit)
    for x_, v in zip(np.arange(4) + (j - 1) * width, vals):
        ax.text(x_, v + 0.005, f"{v:.3f}", ha="center", fontsize=6.5)
ax.set_xticks(np.arange(4), [r[0] for r in gains_tbl])
ax.set(ylabel="gain of the question", ylim=(0, 0.29))
ax.legend(ncols=3, loc="upper right", fontsize=7)
display(fg)
plotlib.save(fg, "t1_gains")

Outlook has the largest gain under all three criteria, so it is the root. Note that the **misclassification error gains
nothing for several questions** (a split can be useful even if no example changes class): entropy and Gini are *smooth*,
strictly concave functions of $p$, so they reward purer children even when the majority class stays the same.

In [ ]:
p = np.linspace(0.001, 0.999, 300)
two = np.c_[p, 1 - p]
plt.plot(p, entropy(two), label="entropy")
plt.plot(p, 2 * gini(two), label="Gini x 2 (rescaled)")
plt.plot(p, 2 * error(two), label="error x 2 (rescaled)")
plt.xlabel("proportion of class 1")
plt.ylabel("impurity")
plt.legend()
plt.title("impurity measures of a two-class node")
plt.show()

# a split on real data where error gains nothing but Gini does
parent = np.array([40, 40])
left, right = np.array([30, 10]), np.array([10, 30])
for name, fn in (("entropy", entropy), ("Gini", gini), ("error", error)):
    print(f"{name:<8} parent {fn(parent):.3f}, children {0.5 * fn(left) + 0.5 * fn(right):.3f}, gain {fn(parent) - 0.5 * fn(left) - 0.5 * fn(right):.3f}")
left, right = np.array([40, 10]), np.array([0, 30])
print("A split with children (40, 10) and (0, 30):")
for name, fn in (("entropy", entropy), ("Gini", gini), ("error", error)):
    print(f"{name:<8} gain {fn(parent) - (50 / 80) * fn(left) - (30 / 80) * fn(right):.3f}")

In [ ]:
# plot t1_curves
slide("The Three Curves")
p_grid = np.linspace(0.001, 0.999, 300)
two_class = np.c_[p_grid, 1 - p_grid]
fg, ax = plotlib.fig("full", 1.5)
ax.plot(p_grid, entropy(two_class), color=BLUE, label="entropy")
ax.plot(p_grid, 2 * gini(two_class), color=GREEN, label="Gini $\\times 2$")
ax.plot(p_grid, 2 * error(two_class), color=ORANGE, label="error $\\times 2$")
ax.set(xlabel="proportion of class 1 in the node", ylabel="impurity", xlim=(0, 1), ylim=(0, 1.05))
ax.legend(loc="lower center", ncols=3, fontsize=7.5)
display(fg)
plotlib.save(fg, "t1_curves")
print(
    "at p = 0.5: entropy",
    float(entropy(np.array([1, 1]))),
    "; Gini x 2",
    float(2 * gini(np.array([1, 1]))),
    "; error x 2",
    float(2 * error(np.array([1, 1]))),
)

### Continuous features: the best threshold

For a numeric feature the candidate questions are "$x \le t$" with $t$ the midpoints between *consecutive sorted values*.
One pass over the sorted data with **cumulative class counts** evaluates every threshold: $O(n \log n)$ for the sort.
The function below is the core of the algorithm; the lab asks you to write it.

In [ ]:
def best_threshold(x, y, n_classes, criterion=gini):
    """Best split `x <= t` of one numeric feature: returns (threshold, gain); (None, 0) when no split helps."""
    order = np.argsort(x, kind="stable")
    xs, ys = x[order], y[order]
    n = len(ys)
    cum = np.cumsum(np.eye(n_classes)[ys], axis=0)           # class counts of the first i+1 examples
    total = cum[-1]
    left, right = cum[:-1], total - cum[:-1]
    n_left = np.arange(1, n)
    gain = criterion(total) - (n_left / n) * criterion(left) - ((n - n_left) / n) * criterion(right)
    gain = np.where(xs[:-1] < xs[1:], gain, -np.inf)          # only between different values
    i = int(np.argmax(gain))
    if not np.isfinite(gain[i]) or gain[i] <= 1e-12:
        return None, 0.0
    return float((xs[i] + xs[i + 1]) / 2), float(gain[i])


rows = []
for j in np.argsort([-best_threshold(Xb[:, j], yb, 2)[1] for j in range(Xb.shape[1])])[:6]:
    t_, g_ = best_threshold(Xb[:, j], yb, 2)
    t2, g2 = best_threshold(Xb[:, j], yb, 2, entropy)
    rows.append((names_b[j], t_, g_, g2))
table(rows, ["breast cancer feature", "best threshold (Gini)", "Gini gain", "entropy gain"])
j = names_b.index("worst_perimeter")
t_, g_ = best_threshold(Xb[:, j], yb, 2)
srt = np.sort(Xb[:, j])
print(f"worst_perimeter: {len(np.unique(srt)) - 1} candidate thresholds, the best is {t_:.1f}")
ths = (srt[:-1] + srt[1:]) / 2
plt.hist([Xb[yb == 1, j], Xb[yb == 0, j]], bins=30, label=["benign", "malignant"], stacked=True)
plt.axvline(t_, color="k", ls="--", label=f"best threshold {t_:.0f}")
plt.xlabel("worst_perimeter")
plt.legend()
plt.show()

In [ ]:
# slide: Entropy or Gini?
slide("Entropy or Gini?")
j_cc = names_b.index("worst_concave_points")
t_g, g_g = best_threshold(Xb[:, j_cc], yb, 2, gini)
t_e, g_e = best_threshold(Xb[:, j_cc], yb, 2, entropy)
print(f"worst_concave_points, all {len(yb)} tumours: best threshold {t_g:.3f} (Gini, gain {g_g:.3f}) and {t_e:.3f} (entropy, gain {g_e:.3f})")

In [ ]:
# slide: Numeric Features: Where to Cut?
slide("Numeric Features: Where to Cut?")
age = np.array([22.0, 25, 31, 38, 45, 52])
bad = np.array([1, 1, 0, 1, 0, 0])
cum = np.cumsum(np.eye(2)[bad], axis=0)  # (good, bad) counts of the first i + 1 applicants
total, left_c = cum[-1], cum[:-1]
right_c = total - left_c
n_left = np.arange(1, 6)
gain_age = gini(total) - n_left / 6 * gini(left_c) - (6 - n_left) / 6 * gini(right_c)
mid_age = (age[:-1] + age[1:]) / 2
thr = pl.DataFrame(
    {
        "threshold": mid_age,
        "left (good, bad)": [f"({int(a)}, {int(b)})" for a, b in left_c],
        "right (good, bad)": [f"({int(a)}, {int(b)})" for a, b in right_c],
        "gain (Gini)": gain_age,
    }
)
show(thr)
print(f"best thresholds: {mid_age[gain_age == gain_age.max()].tolist()} with gain {gain_age.max():.3f} (a tie)")

In [ ]:
# plot t1_thresholds
fg, ax = plotlib.fig("full", 1.4)
ax.bar(mid_age, gain_age, width=2.4, color=BLUE)
for m_, g_ in zip(mid_age, gain_age):
    ax.text(m_, g_ + 0.008, f"{m_:g}: {g_:.3f}", ha="center", fontsize=7)
ax.scatter(age[bad == 1], np.full((bad == 1).sum(), 0.295), color=RED, s=14, label="bad credit (1)", clip_on=False)
ax.scatter(
    age[bad == 0],
    np.full((bad == 0).sum(), 0.295),
    color=GREEN,
    s=14,
    marker="s",
    label="good credit (0)",
    clip_on=False,
)
ax.set(xlabel="age (years): bar = Gini gain of the cut at the midpoint", ylabel="gain", ylim=(0, 0.32), xlim=(20, 54))
ax.legend(loc="center right", bbox_to_anchor=(1.0, 0.62), fontsize=7)
display(fg)
plotlib.save(fg, "t1_thresholds")

## T2 — A tree from scratch

**The algorithm** (CART, Breiman et al. 1984): at a node with examples $S$

1. if $S$ is pure, or the depth limit or the minimum size is reached, make a **leaf** with the majority class;
2. otherwise find the (feature, threshold) with the largest gain over all features;
3. send the examples with $x_j \le t$ to the left child and the rest to the right; recurse.

It is **greedy**: each split is the best *now*, with no look-ahead.

In [ ]:
def fit_tree(X, y, n_classes, max_depth=5, min_leaf=1, criterion=gini, depth=0):
    counts = np.bincount(y, minlength=n_classes)
    node = {"counts": counts, "pred": int(counts.argmax()), "n": len(y)}
    if depth >= max_depth or counts.max() == len(y) or len(y) < 2 * min_leaf:
        return node
    best = (None, None, 0.0)
    for j in range(X.shape[1]):
        t, g = best_threshold(X[:, j], y, n_classes, criterion)
        if t is not None and g > best[2]:
            m = X[:, j] <= t
            if m.sum() >= min_leaf and (~m).sum() >= min_leaf:
                best = (j, t, g)
    if best[0] is None:
        return node
    j, t, g = best
    m = X[:, j] <= t
    node.update(feature=j, threshold=t, gain=g,
                left=fit_tree(X[m], y[m], n_classes, max_depth, min_leaf, criterion, depth + 1),
                right=fit_tree(X[~m], y[~m], n_classes, max_depth, min_leaf, criterion, depth + 1))
    return node


def predict_tree(node, X):
    out = np.empty(len(X), dtype=int)
    for i, x in enumerate(X):
        n = node
        while "feature" in n:
            n = n["left"] if x[n["feature"]] <= n["threshold"] else n["right"]
        out[i] = n["pred"]
    return out


def depth_of(node):
    return 0 if "feature" not in node else 1 + max(depth_of(node["left"]), depth_of(node["right"]))


def leaves_of(node):
    return 1 if "feature" not in node else leaves_of(node["left"]) + leaves_of(node["right"])


rows = []
for name, (X, y) in CLASSIFICATION.items():
    K = len(np.unique(y))
    a, b, ya, yb_ = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
    t0 = time.time()
    ours = fit_tree(a, ya, K, max_depth=6, min_leaf=1)
    t_ours = time.time() - t0
    ref = DecisionTreeClassifier(criterion="gini", max_depth=6, random_state=0).fit(a, ya)
    same = np.mean(predict_tree(ours, b) == ref.predict(b))
    rows.append((name, float(np.mean(predict_tree(ours, b) == yb_)), float(ref.score(b, yb_)), float(same), leaves_of(ours), int(ref.get_n_leaves()), t_ours))
table(rows, ["dataset, depth 6", "acc ours", "acc sklearn", "same pred.", "leaves ours", "leaves sk", "fit (s)"])

The two trees agree on (almost) every test example; the few differences come from **ties**: two features with exactly
the same gain, broken in a different order, and from the random feature order of scikit-learn.

**Reading a tree as rules.** Every leaf is a conjunction of conditions: a rule that a human can read, check and challenge.

In [ ]:
small = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xb, yb)
print(export_text(small, feature_names=names_b, decimals=2))
fig, ax = plt.subplots(figsize=(13, 5))
plot_tree(small, feature_names=names_b, class_names=["malignant", "benign"], filled=True, impurity=True, fontsize=7, ax=ax)
plt.show()

## T3 — Depth, overfitting and pruning

A tree with no limit keeps splitting until every leaf is pure: **100% training accuracy**, and a model that has memorized
the noise (Class 01). Two families of remedies:

* **pre-pruning:** stop early (`max_depth`, `min_samples_leaf`, a minimum gain);
* **post-pruning:** grow fully, then cut back the branches that do not pay for themselves: **cost-complexity pruning**
  minimizes $\ \text{error} + \alpha\cdot\#\text{leaves}$, a ridge-like penalty on the size (Class 03, $\lambda$ again).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
depths = range(1, 16)
rows = []
for ax, (name, (X, y)) in zip(axes, [(k, CLASSIFICATION[k]) for k in ("breast_cancer", "german_credit", "wine (quality >= 6)")]):
    tr_acc, cv_acc, sd = [], [], []
    for d in depths:
        m = DecisionTreeClassifier(max_depth=d, random_state=0)
        tr_acc.append(m.fit(X, y).score(X, y))
        s = cross_val_score(m, X, y, cv=StratifiedKFold(10, shuffle=True, random_state=0))
        cv_acc.append(s.mean())
        sd.append(s.std())
    ax.plot(depths, tr_acc, label="train")
    ax.errorbar(depths, cv_acc, yerr=sd, label="10-fold CV")
    ax.set(xlabel="max_depth", title=name)
    best = int(np.argmax(cv_acc))
    rows.append((name, list(depths)[best], cv_acc[best], tr_acc[-1], cv_acc[-1]))
axes[0].set_ylabel("accuracy")
axes[0].legend()
plt.tight_layout()
plt.show()
table(rows, ["dataset", "best depth", "CV accuracy at best", "train acc, depth 15", "CV acc, depth 15"])

In [ ]:
# slide: Depth and Overfitting
slide("Depth and Overfitting")
Xc, yc = CLASSIFICATION["breast_cancer"]
depth_rows = []
for d_ in range(1, 13):
    tree_ = DecisionTreeClassifier(max_depth=d_, random_state=0)
    train_acc = tree_.fit(Xc, yc).score(Xc, yc)
    cv_scores = cross_val_score(tree_, Xc, yc, cv=StratifiedKFold(10, shuffle=True, random_state=0))
    depth_rows.append(
        {
            "max_depth": d_,
            "train": float(train_acc),
            "validation": float(cv_scores.mean()),
            "sd": float(cv_scores.std()),
        }
    )
depth_tbl = pl.DataFrame(depth_rows)
show(depth_tbl)

In [ ]:
# plot t3_depth
fg, ax = plotlib.fig("full", 1.4)
ax.plot(depth_tbl["max_depth"], depth_tbl["train"], color=BLUE, marker="o", ms=3, label="training")
ax.plot(depth_tbl["max_depth"], depth_tbl["validation"], color=RED, marker="s", ms=3, label="validation (10-fold CV)")
ax.fill_between(
    depth_tbl["max_depth"],
    depth_tbl["validation"] - depth_tbl["sd"],
    depth_tbl["validation"] + depth_tbl["sd"],
    color=RED,
    alpha=0.15,
    linewidth=0,
)
best_d = int(depth_tbl["max_depth"][int(np.argmax(depth_tbl["validation"].to_numpy()))])
ax.axvline(best_d, color=GREY, ls=":", lw=0.9)
ax.annotate(f"best depth {best_d}", (best_d, 0.88), (4, 0), textcoords="offset points", fontsize=7.5, color="#444444")
ax.set(xlabel="max_depth", ylabel="accuracy", ylim=(0.86, 1.01))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2, fontsize=7.5)
display(fg)
plotlib.save(fg, "t3_depth")

**Cost-complexity pruning path** (`ccp_alpha`): each increase of $\alpha$ removes the branch with the smallest
gain per leaf. The path gives the whole family of nested trees at the price of one fit.

In [ ]:
Xa, Xv, ya, yv = train_test_split(Xg, yg, test_size=0.3, stratify=yg, random_state=0)
full = DecisionTreeClassifier(random_state=0).fit(Xa, ya)
path = full.cost_complexity_pruning_path(Xa, ya)
alphas = path.ccp_alphas[::max(1, len(path.ccp_alphas) // 25)]
rows = []
for a_ in alphas:
    m = DecisionTreeClassifier(ccp_alpha=a_, random_state=0).fit(Xa, ya)
    rows.append((float(a_), int(m.get_n_leaves()), m.score(Xa, ya), m.score(Xv, yv)))
table(rows[::2], ["ccp_alpha", "leaves", "train acc", "validation acc"])
best = max(rows, key=lambda r: r[3])
print(f"full tree: {full.get_n_leaves()} leaves, train {full.score(Xa, ya):.3f}, validation {full.score(Xv, yv):.3f}")
print(f"best pruned tree: {best[1]} leaves, validation {best[3]:.3f}  (alpha {best[0]:.4f})")
plt.plot([r[1] for r in rows], [r[2] for r in rows], label="train")
plt.plot([r[1] for r in rows], [r[3] for r in rows], label="validation")
plt.xlabel("number of leaves")
plt.ylabel("accuracy")
plt.legend()
plt.title("german credit: complexity against accuracy")
plt.show()

In [ ]:
# plot t3_pruning
slide("Post-Pruning: Cost-Complexity")
fg, ax = plotlib.fig("full", 1.0)
leaves_ = [r[1] for r in rows]
ax.plot(leaves_, [r[2] for r in rows], color=BLUE, marker="o", ms=2.5, label="training")
ax.plot(leaves_, [r[3] for r in rows], color=RED, marker="s", ms=2.5, label="validation")
best_row = max(rows, key=lambda r: r[3])
ax.annotate(f"{best_row[1]} leaves: {best_row[3]:.3f}", (best_row[1], best_row[3]), (60, 0.64), fontsize=7.5, color=RED, arrowprops={"arrowstyle": "-", "color": RED, "lw": 0.6})
ax.invert_xaxis()
ax.set(xlabel="leaves (pruning goes to the right)", ylabel="accuracy", ylim=(0.6, 1.02))
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.0), ncols=2, fontsize=7.5)
display(fg)
plotlib.save(fg, "t3_pruning")
print(
    f"full tree {full.get_n_leaves()} leaves; 8 leaves or fewer: validation {min(rows, key=lambda r: abs(r[1] - 8))[3]:.3f}"
)

**Other limits.** `min_samples_leaf` (a leaf needs enough examples to be trusted) is the most practical pre-pruning
knob; the table sweeps it together with the impurity criterion.

In [ ]:
rows = []
for name in ("breast_cancer", "german_credit", "heart_disease", "wine (quality >= 6)"):
    X, y = CLASSIFICATION[name]
    for crit in ("gini", "entropy"):
        for leaf in (1, 5, 20, 50):
            s = cross_val_score(DecisionTreeClassifier(criterion=crit, min_samples_leaf=leaf, random_state=0), X, y,
                                cv=StratifiedKFold(10, shuffle=True, random_state=0))
            rows.append((name, crit, leaf, s.mean(), s.std()))
df = pl.DataFrame(rows, schema=["dataset", "criterion", "min_leaf", "CV accuracy", "sd"], orient="row")
print(df.pivot(on="min_leaf", index=["dataset", "criterion"], values="CV accuracy"))

In [ ]:
# plot t3_leaf
slide("Pre-Pruning: the Leaf Size")
fg, axs = plotlib.fig("full", 1.4, ncols=3)
for ax, name in zip(axs, ("breast_cancer", "german_credit", "wine (quality >= 6)")):
    for crit, colour, marker in (("gini", BLUE, "o"), ("entropy", ORANGE, "s")):
        sub = df.filter((pl.col("dataset") == name) & (pl.col("criterion") == crit))
        ax.plot(np.arange(4), sub["CV accuracy"], color=colour, marker=marker, ms=3, label=crit)
    ax.set_xticks(np.arange(4), ["1", "5", "20", "50"])
    ax.set_title(name.replace("_", " "), fontsize=7.5, pad=3)
    ax.set_xlabel("min_samples_leaf", fontsize=7.5)
axs[0].set_ylabel("10-fold CV accuracy")
axs[0].legend(loc="lower left", fontsize=7)
fg.subplots_adjust(wspace=0.4)
display(fg)
plotlib.save(fg, "t3_leaf")

## T4 — Regression trees

For a numeric target the leaf predicts the **mean** of its examples, and a split is judged by the **reduction of the
variance** (the sum of squared errors): the same algorithm with another impurity. The prediction is a **staircase**.

In [ ]:
rng = np.random.default_rng(0)
xs = np.sort(rng.uniform(0, 6, 80))
ys = np.sin(xs) + rng.normal(0, 0.25, 80)
xg = np.linspace(-1, 7, 400)
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
rows = []
for ax, d in zip(axes, (1, 2, 4, 8)):
    m = DecisionTreeRegressor(max_depth=d, random_state=0).fit(xs[:, None], ys)
    ax.scatter(xs, ys, s=8)
    ax.plot(xg, m.predict(xg[:, None]), color="tab:red")
    ax.plot(xg, np.sin(xg), "k:", lw=0.8)
    ax.set_title(f"depth {d}: {m.get_n_leaves()} leaves", fontsize=9)
    rows.append((d, int(m.get_n_leaves()), float(np.sqrt(np.mean((m.predict(xs[:, None]) - ys) ** 2)))))
plt.tight_layout()
plt.show()
table(rows, ["max_depth", "leaves", "train RMSE"])
print("outside the data (x < 0 or x > 6) a tree repeats the value of its last leaf: no extrapolation; the line, however, continues the trend.")

In [ ]:
# slide: The Prediction Is a Staircase
slide("The Prediction Is a Staircase")
tree3 = DecisionTreeRegressor(max_depth=3, random_state=0).fit(xs[:, None], ys)
print(
    f"depth 3: {tree3.get_n_leaves()} leaves, {len(np.unique(tree3.predict(xs[:, None])))} different predicted values"
)
print(
    f"at x = -1 and x = 7 the tree predicts {tree3.predict(np.array([[-1.0], [7.0]])).round(3).tolist()}: the value of the first and the last leaf"
)

In [ ]:
# plot t4_staircase
fg, ax = plotlib.fig("full", 1.5)
ax.scatter(xs, ys, s=7, color=BLUE, zorder=3)
ax.plot(xg, tree3.predict(xg[:, None]), color=RED, lw=1.5, label=f"tree, depth 3 ({tree3.get_n_leaves()} leaves)")
ax.plot(xg, np.sin(xg), color="#444444", ls=":", lw=1, label="true curve")
ax.set(xlabel="$x$", ylabel="$y$", xlim=(-1, 7))
ax.legend(loc="lower left", fontsize=7.5)
display(fg)
plotlib.save(fg, "t4_staircase")

**A split by hand.** Five points $(x, y)$: $(1,1.0), (2,1.2), (3,3.1), (4,3.0), (5,3.2)$. The best threshold is the one with the
smallest total squared error around the two means.

In [ ]:
x5 = np.array([1.0, 2, 3, 4, 5])
y5 = np.array([1.0, 1.2, 3.1, 3.0, 3.2])
print(f"no split: mean {y5.mean():.2f}, SSE {np.sum((y5 - y5.mean()) ** 2):.3f}")
rows = []
for t in (1.5, 2.5, 3.5, 4.5):
    L, R = y5[x5 <= t], y5[x5 > t]
    sse = np.sum((L - L.mean()) ** 2) + np.sum((R - R.mean()) ** 2)
    rows.append((t, float(L.mean()), float(R.mean()), float(sse)))
table(rows, ["threshold", "left mean", "right mean", "total SSE"])

In [ ]:
# slide: A Regression Split by Hand
slide("A Regression Split by Hand")
sse_all = float(np.sum((y5 - y5.mean()) ** 2))
sse_by_t = {
    t: float(np.sum((y5[x5 <= t] - y5[x5 <= t].mean()) ** 2) + np.sum((y5[x5 > t] - y5[x5 > t].mean()) ** 2))
    for t in (1.5, 2.5, 3.5, 4.5)
}
best_t = min(sse_by_t, key=lambda t: sse_by_t[t])
print(
    f"no split: mean {y5.mean():.2f}, SSE {sse_all:.3f}; best threshold {best_t} with SSE {sse_by_t[best_t]:.3f}; gain {sse_all - sse_by_t[best_t]:.3f}"
)

In [ ]:
# plot t4_sse
fg, axs = plotlib.fig("full", 1.1, ncols=2, sharey=True)
for ax, title, groups_ in (
    (axs[0], f"no split: mean {y5.mean():.2f}, SSE {sse_all:.3f}", [(x5 > 0)]),
    (axs[1], f"split at {best_t}: SSE {sse_by_t[best_t]:.3f}", [x5 <= best_t, x5 > best_t]),
):
    for g in groups_:
        mu = y5[g].mean()
        ax.hlines(mu, x5[g].min() - 0.4, x5[g].max() + 0.4, color=RED, lw=1.4)
        ax.vlines(x5[g], y5[g], mu, color=GREY, lw=0.9)
    ax.scatter(x5, y5, s=12, color=BLUE, zorder=3)
    ax.set_title(title, fontsize=7.5, pad=3)
    ax.set(xlabel="$x$", xlim=(0.4, 5.6))
axs[0].set_ylabel("$y$")
display(fg)
plotlib.save(fg, "t4_sse")

In [ ]:
# plot t4_thresholds
fg, ax = plotlib.fig("full", 1.3)
ts_ = list(sse_by_t)
ax.bar(ts_, [sse_by_t[t] for t in ts_], width=0.6, color=[RED if t == best_t else BLUE for t in ts_])
for t in ts_:
    ax.text(t, sse_by_t[t] + 0.08, f"{sse_by_t[t]:.3f}", ha="center", fontsize=7.5)
ax.axhline(sse_all, color="#444444", ls=":", lw=0.9)
ax.annotate(f"no split: {sse_all:.3f}", (4.9, sse_all), (0, 3), textcoords="offset points", fontsize=7.5, ha="right")
ax.set(xlabel="threshold", ylabel="total SSE", ylim=(0, 5.6), xticks=ts_)
display(fg)
plotlib.save(fg, "t4_thresholds")

**Real data, and the extrapolation failure.** Four regression sets, 5-fold RMSE of the baseline (the mean), a linear
model, and trees of increasing depth:

In [ ]:
rows = []
for name, target in (("concrete", "strength"), ("abalone", "rings"), ("auto_mpg", "mpg"), ("california_housing", "MedHouseVal")):
    X, y, _ = load_california() if name == "california_housing" else load_table(name, target)
    y = y.astype(float)
    if len(y) > 6000:
        idx = np.random.default_rng(0).choice(len(y), 6000, replace=False)
        X, y = X[idx], y[idx]
    res = {}
    for label, mk in (("mean", lambda: DecisionTreeRegressor(max_depth=0 + 1, min_samples_leaf=len(y))), ("linear", LinearRegression),
                      ("tree d=3", lambda: DecisionTreeRegressor(max_depth=3, random_state=0)),
                      ("tree d=6", lambda: DecisionTreeRegressor(max_depth=6, random_state=0)),
                      ("tree d=12", lambda: DecisionTreeRegressor(max_depth=12, random_state=0)),
                      ("tree leaf>=10", lambda: DecisionTreeRegressor(min_samples_leaf=10, random_state=0))):
        errs = []
        for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, np.digitize(y, np.quantile(y, [0.25, 0.5, 0.75]))):
            m = mk().fit(X[tr], y[tr])
            errs.append(np.sqrt(np.mean((m.predict(X[te]) - y[te]) ** 2)))
        res[label] = float(np.mean(errs))
    rows.append((name, *res.values()))
table(rows, ["RMSE (5-fold)", "mean", "linear", "tree d=3", "tree d=6", "tree d=12", "tree leaf>=10"])

*Extrapolation:* train on the part of California housing with the lowest median income and test on the highest.
A linear model continues the trend; a tree predicts the value of the last leaf.

In [ ]:
Xh2, yh2, nh = load_california()
j = nh.index("MedInc")
cut = np.quantile(Xh2[:, j], 0.7)
tr, te = Xh2[:, j] <= cut, Xh2[:, j] > cut
rows = []
for label, m in (("mean", DecisionTreeRegressor(max_depth=1, min_samples_leaf=len(yh2))), ("linear", LinearRegression()), ("tree d=6", DecisionTreeRegressor(max_depth=6, random_state=0)),
                 ("tree leaf>=20", DecisionTreeRegressor(min_samples_leaf=20, random_state=0))):
    m.fit(Xh2[tr], yh2[tr])
    inside = np.sqrt(np.mean((m.predict(Xh2[tr]) - yh2[tr]) ** 2))
    outside = np.sqrt(np.mean((m.predict(Xh2[te]) - yh2[te]) ** 2))
    rows.append((label, float(inside), float(outside)))
table(rows, ["model", "RMSE, seen range", "RMSE, richest 30%"])

In [ ]:
# plot t4_extrap
slide("A Tree Cannot Extrapolate")
fg, ax = plotlib.fig("full", 1.4)
names_m = [r[0] for r in rows]
for j, (lab, colour) in enumerate((("seen range", BLUE), ("richest 30% (never seen)", RED))):
    vals = [r[1 + j] for r in rows]
    xpos = np.arange(len(rows)) + (j - 0.5) * 0.36
    ax.bar(xpos, vals, 0.36, color=colour, label=lab)
    for x_, v in zip(xpos, vals):
        ax.text(x_, v + 0.03, f"{v:.3f}", ha="center", fontsize=6.5)
ax.set_xticks(np.arange(len(rows)), names_m)
ax.set(ylabel="RMSE", ylim=(0, 2.5))
ax.legend(loc="upper left", fontsize=7.5, ncols=2)
display(fg)
plotlib.save(fg, "t4_extrap")

## T5 — Instability and importances

A small change of the data can change the **first split**, and then everything below it: a tree has **high variance**
(Class 01). 30 trees, each trained on a different bootstrap sample (draw $n$ examples with replacement) of the breast cancer
data:

In [ ]:
rng = np.random.default_rng(0)
Xa, Xt, ya, yt = train_test_split(Xb, yb, test_size=0.3, stratify=yb, random_state=0)
roots, accs, leaves = [], [], []
for b in range(30):
    idx = rng.integers(0, len(ya), len(ya))
    m = DecisionTreeClassifier(random_state=b).fit(Xa[idx], ya[idx])
    roots.append(names_b[m.tree_.feature[0]])
    accs.append(m.score(Xt, yt))
    leaves.append(m.get_n_leaves())
vals, cnt = np.unique(roots, return_counts=True)
print("feature used at the root over 30 bootstrap trees:", dict(zip(vals.tolist(), cnt.tolist())))
print(f"test accuracy of the 30 trees: mean {np.mean(accs):.3f}, sd {np.std(accs):.3f}, min {np.min(accs):.3f}, max {np.max(accs):.3f}; leaves {int(np.min(leaves))}-{int(np.max(leaves))}")
votes = np.mean([DecisionTreeClassifier(random_state=b).fit(Xa[i], ya[i]).predict(Xt) for b, i in enumerate([rng.integers(0, len(ya), len(ya)) for _ in range(30)])], axis=0)
print(f"majority vote of the 30 trees (bagging, Class 07): test accuracy {np.mean((votes > 0.5) == yt):.3f}")

In [ ]:
# plot t5_root
slide("Instability")
fg, ax = plotlib.fig("full", 1.3)
order_r = np.argsort(-cnt)
ax.bar(np.arange(len(vals)), cnt[order_r], color=BLUE, width=0.6)
for x_, c_ in enumerate(cnt[order_r]):
    ax.text(x_, c_ + 0.4, str(int(c_)), ha="center", fontsize=7.5)
ax.set_xticks(np.arange(len(vals)), [vals[i].replace("_", "\n") for i in order_r], fontsize=6.5)
ax.set(ylabel="trees (of 30)", ylim=(0, 23))
ax.set_title(
    f"feature at the root; test accuracy of the 30 trees: mean {np.mean(accs):.3f}, sd {np.std(accs):.3f}",
    fontsize=7.5,
    pad=3,
)
display(fg)
plotlib.save(fg, "t5_root")

**Feature importances.** A tree gives a free ranking: the total impurity decrease of the splits on each feature. But it
is **biased towards features with many distinct values** (more candidate thresholds, more chances to fit noise). Add one
*random* continuous feature and one random binary feature to the wine data and compare with the permutation importance
(the drop of the test accuracy when the feature is shuffled) on held-out data:

In [ ]:
rng = np.random.default_rng(1)
Xn = np.c_[Xw, rng.normal(size=len(yw)), rng.integers(0, 2, len(yw))]
names_n = names_w + ["RANDOM continuous", "RANDOM binary"]
Xa, Xt, ya, yt = train_test_split(Xn, yw, test_size=0.3, stratify=yw, random_state=0)
m = DecisionTreeClassifier(min_samples_leaf=5, random_state=0).fit(Xa, ya)
perm = permutation_importance(m, Xt, yt, n_repeats=20, random_state=0)
order = np.argsort(-m.feature_importances_)
rows = [(names_n[i], float(m.feature_importances_[i]), float(perm.importances_mean[i])) for i in order[:8]]
rows += [(n_, float(m.feature_importances_[names_n.index(n_)]), float(perm.importances_mean[names_n.index(n_)])) for n_ in ("RANDOM continuous", "RANDOM binary")]
table(rows, ["feature", "impurity importance (train)", "permutation importance (test)"])
print(f"test accuracy {m.score(Xt, yt):.3f}; the random continuous feature gets {m.feature_importances_[-2]:.3f} of the impurity importance")

In [ ]:
# plot t5_importances
slide("Feature Importances and Their Bias")
fg, ax = plotlib.fig("full", 1.5)
sel = [*order[:3], names_n.index("RANDOM continuous"), names_n.index("RANDOM binary")]
ypos = np.arange(len(sel))[::-1]
ax.barh(ypos + 0.18, [m.feature_importances_[i] for i in sel], 0.34, color=BLUE, label="impurity importance (train)")
ax.barh(ypos - 0.18, [perm.importances_mean[i] for i in sel], 0.34, color=ORANGE, label="permutation importance (test)")
for y_, i in zip(ypos, sel):
    ax.text(
        max(m.feature_importances_[i], 0) + 0.005,
        y_ + 0.18,
        f"{m.feature_importances_[i]:.3f}",
        va="center",
        fontsize=7,
    )
    ax.text(
        max(perm.importances_mean[i], 0) + 0.005, y_ - 0.18, f"{perm.importances_mean[i]:.3f}", va="center", fontsize=7
    )
ax.set_yticks(ypos, [names_n[i].replace("_", " ") for i in sel])
ax.set(xlabel="importance", xlim=(-0.01, 0.33))
ax.legend(loc="lower right", fontsize=7)
display(fg)
plotlib.save(fg, "t5_importances")

**Correlated features** share the credit: duplicate the most important feature and the importance of each copy halves.

In [ ]:
j = int(np.argmax(m.feature_importances_))
Xdup = np.c_[Xn, Xn[:, j]]
a2, t2, ya2, yt2 = train_test_split(Xdup, yw, test_size=0.3, stratify=yw, random_state=0)
m2 = DecisionTreeClassifier(min_samples_leaf=5, random_state=0).fit(a2, ya2)
print(f"{names_n[j]}: importance {m.feature_importances_[j]:.3f} alone; with a copy: {m2.feature_importances_[j]:.3f} + {m2.feature_importances_[-1]:.3f} = {m2.feature_importances_[j] + m2.feature_importances_[-1]:.3f}")

## T6 — Feature types and encoding

A column is not just a number. Its **type** says which questions and which distances make sense: a tree asks
"is the value in this subset?", k-NN adds up differences. Here the columns of five data sets are typed by hand
(a program cannot tell an ordinal column from a nominal one), five encodings are compared on a tree and on k-NN with a
spread over 25 splits, and a high-cardinality column and missing values are the failure cases.

In [ ]:
# slide: Why the Type of a Column Matters
slide("Why the Type of a Column Matters")
# types that no program can read from the file: decided from the data description (datasets/README.md)
ORDER = {  # ordinal columns, in their natural order (levels outside the scale come last)
    "checking_status": ["<0", "0<=X<200", ">=200", "no checking"],
    "savings_status": ["<100", "100<=X<500", "500<=X<1000", ">=1000", "no known savings"],
    "employment": ["unemployed", "<1", "1<=X<4", "4<=X<7", ">=7"],
    "job": ["unemp/unskilled non res", "unskilled resident", "skilled", "high qualif/self emp/mgmt"],
    "slope": ["down", "flat", "up"],
}
INT_CODED_NOMINAL = {"origin"}  # a category stored as the numbers 1, 2, 3 (US, Europe, Japan)
TARGETS = {
    "german_credit": "class",
    "heart_disease": "num",
    "auto_mpg": "mpg",
    "abalone": "rings",
    "wine_quality": "quality",
}


def column_type(df, col):
    """continuous, discrete, binary, ordinal or nominal (a rule of thumb plus the two dictionaries above)."""
    s = df[col]
    if col in ORDER:
        return "ordinal"
    if s.dtype == pl.String or col in INT_CODED_NOMINAL:
        return "binary" if s.n_unique() == 2 else "nominal"
    return "discrete" if s.drop_nulls().n_unique() <= 20 else "continuous"


TYPES = ["continuous", "discrete", "binary", "ordinal", "nominal"]
type_rows = []
frames6 = {name: pl.read_csv(DATA / f"{name}.csv.zst") for name in TARGETS}
for name, target in TARGETS.items():
    df = frames6[name].drop(target)
    kinds = [column_type(df, c) for c in df.columns]
    type_rows.append({"data set": name, "columns": len(kinds), **{t: kinds.count(t) for t in TYPES}})
types_table = pl.DataFrame(type_rows)
print(types_table)
for name in ("german_credit", "heart_disease", "auto_mpg"):
    df = frames6[name].drop(TARGETS[name])
    print(name, {t: [c for c in df.columns if column_type(df, c) == t] for t in TYPES if t not in ("continuous",)})

In [ ]:
# plot t6_types
fg, ax = plotlib.fig("full", 1.25)
left = np.zeros(len(types_table))
for t, colour in zip(TYPES, (BLUE, SKY, GREY, ORANGE, RED)):
    vals = types_table[t].to_numpy()
    ax.barh(types_table["data set"], vals, left=left, color=colour, label=t, height=0.62)
    for y_, (v, l_) in enumerate(zip(vals, left)):
        if v:
            ax.text(
                l_ + v / 2,
                y_,
                str(v),
                ha="center",
                va="center",
                fontsize=7,
                color="white" if colour in (BLUE, RED, GREY) else "black",
            )
    left += vals
ax.invert_yaxis()
ax.set(xlabel="number of feature columns")
ax.grid(False)
ax.legend(ncols=5, loc="upper center", bbox_to_anchor=(0.45, 1.25), fontsize=7.5, handlelength=1, columnspacing=0.9)
display(fg)
plotlib.save(fg, "t6_types")

In [ ]:
# slide: Encoding a Category as Numbers
slide("Encoding a Category as Numbers")
german = frames6["german_credit"]
housing = german.select("housing", (pl.col("class") == "bad").cast(pl.Int32).alias("bad"))
levels = sorted(housing["housing"].unique().to_list())
freq = housing["housing"].value_counts().with_columns((pl.col("count") / len(housing)).alias("frequency"))
bad_rate = housing.group_by("housing").agg(pl.col("bad").mean().alias("bad rate"))
enc = (
    pl.DataFrame({"housing": levels, "ordinal (alphabetical)": list(range(len(levels)))})
    .join(freq.select("housing", "frequency"), on="housing")
    .join(bad_rate, on="housing")
)
for lv in levels:
    enc = enc.with_columns((pl.col("housing") == lv).cast(pl.Int32).alias(f"one-hot: {lv}"))
show(enc, 3)
print(
    f"one-hot turns 1 column of {len(levels)} levels into {len(levels)} columns; frequency and target (bad rate) encodings keep 1 column"
)

In [ ]:
# slide: A Tree Asks: Is the Value in a Subset?
slide("A Tree Asks: Is the Value in a Subset?")


def n_subset_splits(c):
    """Binary questions 'value in S' on c levels: 2^c subsets, minus the empty and the full set, halved (S and its complement ask the same)."""
    return 2 ** (c - 1) - 1


def best_subset_gini(counts):
    """Exhaustive search of the best question 'level in S' by Gini gain; counts is (c, 2) = class counts per level."""
    c = len(counts)
    codes = np.arange(1, 2 ** (c - 1))  # fix the last level outside S: every split once
    member = ((codes[:, None] >> np.arange(c)) & 1).astype(float)  # (splits, c)
    left = member @ counts
    right = counts.sum(0) - left
    nl, nr = left.sum(1), right.sum(1)
    gini = lambda v, n: 1 - ((v / n[:, None]) ** 2).sum(1)  # noqa: E731
    total = counts.sum(0)
    parent = 1 - ((total / total.sum()) ** 2).sum()
    gain = parent - nl / (nl + nr) * gini(left, nl) - nr / (nl + nr) * gini(right, nr)
    return float(gain.max()), len(codes)


def sorted_gini(counts):
    """Sorting the levels by the share of class 1 leaves only c - 1 candidate splits (Fisher 1958; CART, two classes)."""
    order = np.argsort(counts[:, 1] / counts.sum(1))
    cum = np.cumsum(counts[order], axis=0)[:-1]
    left, right = cum, counts.sum(0) - cum
    nl, nr = left.sum(1), right.sum(1)
    total = counts.sum(0)
    parent = 1 - ((total / total.sum()) ** 2).sum()
    gain = (
        parent
        - nl / (nl + nr) * (1 - ((left / nl[:, None]) ** 2).sum(1))
        - nr / (nl + nr) * (1 - ((right / nr[:, None]) ** 2).sum(1))
    )
    return float(gain.max()), len(cum)


rows = []
for col in ("housing", "checking_status", "purpose"):
    cnt = german.group_by(col, "class").len().pivot(on="class", index=col, values="len").fill_null(0).sort(col)
    counts = cnt.select("bad", "good").to_numpy().astype(float)
    g_all, n_all = best_subset_gini(counts)
    g_sorted, n_sorted = sorted_gini(counts)
    rows.append(
        {
            "column": col,
            "levels v": len(counts),
            "2^(v-1) - 1": n_all,
            "v - 1": n_sorted,
            "best gain, all subsets": g_all,
            "best gain, sorted": g_sorted,
        }
    )
show(pl.DataFrame(rows), 4)
print("c levels give 2^(c-1) - 1 questions:", {c: n_subset_splits(c) for c in (2, 3, 4, 10, 20, 30)})
rng = np.random.default_rng(6)
times = []
for c in (8, 12, 16, 20):
    counts = rng.integers(5, 60, size=(c, 2)).astype(float)
    t0 = time.perf_counter()
    best_subset_gini(counts)
    t_all = time.perf_counter() - t0
    t0 = time.perf_counter()
    sorted_gini(counts)
    times.append(
        {
            "c": c,
            "questions tried": n_subset_splits(c),
            "all subsets (s)": t_all,
            "sorted (s)": time.perf_counter() - t0,
        }
    )
show(pl.DataFrame(times), 5)
print("timings are typical values of this machine")

In [ ]:
# plot t6_splits
fg, ax = plotlib.fig("full", 1.3)
cs = np.arange(2, 31)
ax.plot(cs, [n_subset_splits(c) for c in cs], color=RED, label="all subsets: $2^{v-1} - 1$")
ax.plot(cs, cs - 1, color=GREEN, label="sorted by class share: $v - 1$")
for c, text, dy in (
    (3, "$v$ = 3: 3", 10),
    (10, "$v$ = 10: 511", 12),
    (20, "$v$ = 20: 524 287", 12),
    (30, "$v$ = 30: 536 870 911", 8),
):
    ax.plot([c], [n_subset_splits(c)], "o", color=RED, ms=3)
    ax.annotate(
        text,
        (c, n_subset_splits(c)),
        (-3, dy),
        textcoords="offset points",
        fontsize=7.5,
        color=RED,
        ha="right" if c > 10 else "left",
    )
ax.set(
    yscale="log",
    xlabel="number of levels $v$ of the category",
    ylabel="questions (log scale)",
    xlim=(2, 31),
    ylim=(0.8, 4e9),
)
ax.legend(loc="center right", bbox_to_anchor=(1.0, 0.3), fontsize=7.5)
display(fg)
plotlib.save(fg, "t6_splits")

In [ ]:
# slide: KNN Needs a Distance Between Categories
slide("KNN Needs a Distance Between Categories")
col = "checking_status"
natural = ORDER[col]
alpha = sorted(natural)
share = german[col].value_counts().with_columns((pl.col("count") / len(german)).alias("f"))
freq_of = dict(zip(share[col].to_list(), share["f"].to_list()))
codes = {
    "ordinal, alphabetical": {lv: i for i, lv in enumerate(alpha)},
    "ordinal, natural order": {lv: i for i, lv in enumerate(natural)},
}


def pair_distance(code_a, code_b, kind):
    if kind == "one-hot":
        return 0.0 if code_a == code_b else float(np.sqrt(2))
    return abs(code_a - code_b)


dist = {}
for name, cmap in codes.items():
    dist[name] = np.array([[abs(cmap[a] - cmap[b]) for b in natural] for a in natural], float)
dist["one-hot"] = np.array([[0.0 if a == b else np.sqrt(2) for b in natural] for a in natural])
dist["frequency"] = np.array([[abs(freq_of[a] - freq_of[b]) for b in natural] for a in natural])
for name, m in dist.items():
    print(
        f"{name}: "
        + "; ".join(f"{natural[i]} - {natural[j]} = {m[i, j]:.3f}" for i in range(4) for j in range(i + 1, 4))
    )
print("levels:", natural, "frequencies:", {k: round(v, 3) for k, v in freq_of.items()})


def gower(a, b, df, ranges):
    """Gower distance of two rows: mean over the columns of |a - b| / range (numeric), 0 or 1 (categories)."""
    parts = []
    for c in df.columns:
        if c in ranges:
            parts.append(abs(a[c] - b[c]) / ranges[c])
        else:
            parts.append(0.0 if a[c] == b[c] else 1.0)
    return float(np.mean(parts)), parts


feat = german.drop("class")
ranges = {c: float(feat[c].max() - feat[c].min()) for c in feat.columns if feat[c].dtype != pl.String}
row_a, row_b = feat.row(0, named=True), feat.row(1, named=True)
g, parts = gower(row_a, row_b, feat, ranges)
print(
    f"Gower distance of applicants 1 and 2 of German credit: {g:.3f} = mean of {len(parts)} per-column terms; {sum(p == 1.0 for p in parts)} categories differ, largest numeric term {max(p for c, p in zip(feat.columns, parts) if c in ranges):.3f}"
)
euclid = float(
    np.sqrt(
        sum(
            ((row_a[c] - row_b[c]) / ranges[c]) ** 2 if c in ranges else (0.0 if row_a[c] == row_b[c] else 1.0)
            for c in feat.columns
        )
    )
)
print(f"for comparison, the Euclidean distance after min-max scaling and one-hot: {euclid:.3f}")

In [ ]:
# plot t6_distances
fg, axs = plotlib.fig("full", 1.6, ncols=3)
short = ["<0", "0-200", ">=200", "none"]
for ax, (name, m) in zip(axs, {k: v for k, v in dist.items() if k != "ordinal, alphabetical"}.items()):
    ax.imshow(m, cmap="Blues", vmin=0, vmax=3.0)
    for i in range(4):
        for j in range(4):
            ax.text(
                j,
                i,
                f"{m[i, j]:.2f}".rstrip("0").rstrip("."),
                ha="center",
                va="center",
                fontsize=7,
                color="white" if m[i, j] > 1.7 else "black",
            )
    ax.set(xticks=range(4), yticks=range(4))
    ax.set_xticklabels(short, fontsize=6.5, rotation=60)
    ax.set_yticklabels(short if ax is axs[0] else [], fontsize=6.5)
    ax.set_title(name.replace(", ", ",\n") if "," in name else name + "\n", fontsize=7.5, pad=3)
    ax.grid(False)
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
display(fg)
plotlib.save(fg, "t6_distances")

In [ ]:
# slide: Does the Encoding Change the Accuracy?
slide("Does the Encoding Change the Accuracy?")
RECIPES = ["ordinal, alphabetical", "ordinal, natural order", "one-hot", "frequency", "target"]


class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Replace each category by its relative frequency in the training data (unseen categories get 0)."""

    def fit(self, X, y=None):
        self.freq_ = [dict(zip(*np.unique(X[:, j].astype(str), return_counts=True))) for j in range(X.shape[1])]
        self.n_ = len(X)
        return self

    def transform(self, X):
        return np.column_stack([[self.freq_[j].get(str(v), 0) / self.n_ for v in X[:, j]] for j in range(X.shape[1])])


def table_xy(name):
    """Features as an object array (numbers as floats, categories as text; a missing category is its own level), target, column types."""
    df = frames6[name]
    target = TARGETS[name]
    feats = df.drop(target)
    cat = [c for c in feats.columns if feats[c].dtype == pl.String or c in INT_CODED_NOMINAL]
    feats = feats.with_columns([pl.col(c).cast(pl.String).fill_null("missing") for c in cat])
    num = [c for c in feats.columns if c not in cat]
    feats = feats.with_columns([pl.col(c).fill_null(pl.col(c).median()) for c in num])
    y = df[target].to_numpy()
    y = (y == y.max()).astype(int) if y.dtype == object or df[target].dtype == pl.String else y.astype(float)
    if name == "german_credit":
        y = (df[target].to_numpy() == "bad").astype(int)
    if name == "heart_disease":
        y = (df[target].to_numpy() == ">50_1").astype(int)
    return feats, y, num, cat


def make_model(kind, recipe, feats, num, cat, k=9):
    """Encoder + model; trees see the codes as they are, k-NN scales the numeric and the coded columns (never the 0/1 one-hot columns)."""
    cols = list(feats.columns)
    num_i, cat_i = [cols.index(c) for c in num], [cols.index(c) for c in cat]
    ordinal_cols = [c for c in cat if c in ORDER]
    scale = StandardScaler() if kind == "knn" else "passthrough"
    if recipe == "one-hot":
        cat_part = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    elif recipe == "frequency":
        cat_part = make_pipeline(FrequencyEncoder(), scale)
    elif recipe == "target":
        cat_part = make_pipeline(TargetEncoder(random_state=0), scale)
    else:
        cats = []
        for c in cat:
            lv = sorted(set(feats[c].to_list()))
            cats.append(
                [x for x in ORDER[c] if x in lv] + [x for x in lv if x not in ORDER[c]]
                if recipe == "ordinal, natural order" and c in ordinal_cols
                else lv
            )
        cat_part = make_pipeline(
            OrdinalEncoder(categories=cats, handle_unknown="use_encoded_value", unknown_value=-1), scale
        )
    ct = ColumnTransformer([("num", scale, num_i), ("cat", cat_part, cat_i)])
    return ct, cols


def score_encodings(name, kind, n_repeats=5, k=9):
    """Mean and spread (sd) of 25 scores: 5-fold CV repeated 5 times; accuracy for classes, RMSE for numbers."""
    feats, y, num, cat = table_xy(name)
    X = feats.to_numpy()
    classification = name in ("german_credit", "heart_disease")
    cv = (
        RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=6)
        if classification
        else RepeatedKFold(n_splits=5, n_repeats=n_repeats, random_state=6)
    )
    out = []
    for recipe in RECIPES:
        ct, _ = make_model(kind, recipe, feats, num, cat)
        if classification:
            est = (
                DecisionTreeClassifier(min_samples_leaf=5, random_state=0)
                if kind == "tree"
                else KNeighborsClassifier(k)
            )
        else:
            est = (
                DecisionTreeRegressor(min_samples_leaf=5, random_state=0) if kind == "tree" else KNeighborsRegressor(k)
            )
        pipe = make_pipeline(ct, est)
        scoring = "accuracy" if classification else "neg_root_mean_squared_error"
        s = cross_val_score(pipe, X, y, cv=cv, scoring=scoring)
        s = s if classification else -s
        out.append(
            {
                "data set": name,
                "model": kind,
                "encoding": recipe,
                "mean": float(s.mean()),
                "sd": float(s.std()),
                "columns after encoding": int(pipe[:-1].fit(X, y).transform(X).shape[1]),
            }
        )
    return out


t0 = time.time()
enc_rows = []
for name in ("german_credit", "heart_disease", "auto_mpg", "abalone"):
    for kind in ("tree", "knn"):
        enc_rows += score_encodings(name, kind)
enc_scores = pl.DataFrame(enc_rows)
for name in ("german_credit", "heart_disease", "auto_mpg", "abalone"):
    metric = "accuracy" if name in ("german_credit", "heart_disease") else "RMSE"
    print(f"{name} ({metric}, mean and sd over 25 splits)")
    wide = (
        enc_scores.filter(pl.col("data set") == name)
        .with_columns(pl.format("{} ({})", pl.col("mean").round(3), pl.col("sd").round(3)).alias("score"))
        .pivot(on="model", index="encoding", values="score")
    )
    print(wide.sort(pl.col("encoding").replace_strict(RECIPES, list(range(5)))))
print(
    "columns after encoding, German credit: "
    + ", ".join(f"{r['encoding']} {r['columns after encoding']}" for r in enc_rows[:5])
)
print(f"cell time {time.time() - t0:.0f} s")

In [ ]:
# plot t6_accuracy
fg, axs = plotlib.fig("full", 1.4, ncols=4, sharey=True)
for ax, name, label in zip(
    axs,
    ("german_credit", "heart_disease", "auto_mpg", "abalone"),
    ("accuracy", "accuracy", "RMSE (mpg)", "RMSE (rings)"),
):
    for kind, colour, dy in (("tree", GREEN, -0.13), ("knn", BLUE, 0.13)):
        sub = enc_scores.filter((pl.col("data set") == name) & (pl.col("model") == kind))
        pos = np.array([RECIPES.index(e) for e in sub["encoding"]]) + dy
        ax.errorbar(
            sub["mean"],
            pos,
            xerr=sub["sd"],
            fmt="o",
            color=colour,
            ms=3,
            elinewidth=0.9,
            capsize=1.5,
            label="tree" if kind == "tree" else "k-NN",
        )
    ax.set(xlabel=label, yticks=range(5))
    ax.set_title(name.replace("_", " "), fontsize=7.5, pad=3)
    ax.tick_params(axis="x", labelsize=6.5)
    ax.locator_params(axis="x", nbins=2)
axs[0].set_yticklabels(RECIPES, fontsize=7)
axs[0].invert_yaxis()
axs[0].legend(loc="lower left", bbox_to_anchor=(0.0, 1.12), ncols=2, fontsize=7.5)
display(fg)
plotlib.save(fg, "t6_accuracy")

In [ ]:
# slide: Many Levels and Missing Values
slide("Many Levels and Missing Values")
LEVELS = [0, 2, 10, 50, 200, 800]


def add_postcode(feats, c, seed):
    """A made-up postcode column with c levels, unrelated to the class (pure noise)."""
    rng = np.random.default_rng(seed)
    return feats.with_columns(pl.Series("postcode", [f"P{i:03d}" for i in rng.integers(0, c, len(feats))]))


def cardinality_scores(c, n_repeats=5):
    feats, y, num, cat = table_xy("german_credit")
    if c:
        feats = add_postcode(feats, c, seed=6)
        cat = [*cat, "postcode"]
    X = feats.to_numpy()
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=6)
    res = []
    for label, kind, recipe, scaled in (
        ("tree, one-hot", "tree", "one-hot", False),
        ("tree, ordinal", "tree", "ordinal, alphabetical", False),
        ("tree, target", "tree", "target", False),
        ("k-NN, one-hot unscaled", "knn", "one-hot", False),
        ("k-NN, one-hot standardized", "knn", "one-hot", True),
    ):
        ct, _ = make_model(kind, recipe, feats, num, cat)
        steps = [ct] + ([StandardScaler()] if scaled else [])
        est = DecisionTreeClassifier(min_samples_leaf=5, random_state=0) if kind == "tree" else KNeighborsClassifier(9)
        pipe = make_pipeline(*steps, est)
        s = cross_val_score(pipe, X, y, cv=cv)
        res.append(
            {
                "levels v": c,
                "model": label,
                "mean": float(s.mean()),
                "sd": float(s.std()),
                "columns": int(make_pipeline(*steps).fit(X, y).transform(X).shape[1]),
            }
        )
    return res


t0 = time.time()
card = pl.DataFrame([r for c in LEVELS for r in cardinality_scores(c)])
print(card.pivot(on="model", index="levels v", values="mean").with_columns(pl.col(pl.Float64).round(3)))
print(
    "columns after one-hot:",
    dict(
        zip(
            card.filter(pl.col("model") == "tree, one-hot")["levels v"],
            card.filter(pl.col("model") == "tree, one-hot")["columns"],
        )
    ),
)
print(f"cell time {time.time() - t0:.0f} s")

In [ ]:
# slide: Many Levels and Missing Values (missing values)
MISSING = [0.0, 0.1, 0.2, 0.3, 0.4]
heart = frames6["heart_disease"]
print(
    f"heart disease: {int(heart.null_count().sum_horizontal()[0])} missing values in {len(heart)} rows (columns: {[c for c in heart.columns if heart[c].null_count()]}); {len(heart.drop_nulls())} rows are complete"
)
wine = frames6["wine_quality"]
Xwine = wine.drop("quality").to_numpy()
ywine = (wine["quality"].to_numpy() >= 6).astype(int)
for p in MISSING[1:]:
    print(
        f"wine, p = {p:.1f} of the values missing in each of 11 columns: complete rows left {(1 - p) ** Xwine.shape[1]:.3f} of the data"
    )


def missing_scores(p, n_masks=4):
    """Remove a share p of the values of every column of the wine data (completely at random) and compare four ways to cope."""
    models = {
        "tree, native NaN": lambda: DecisionTreeClassifier(min_samples_leaf=5, random_state=0),
        "tree, median": lambda: make_pipeline(
            SimpleImputer(strategy="median"), DecisionTreeClassifier(min_samples_leaf=5, random_state=0)
        ),
        "tree, median + flag": lambda: make_pipeline(
            SimpleImputer(strategy="median", add_indicator=True),
            DecisionTreeClassifier(min_samples_leaf=5, random_state=0),
        ),
        "k-NN, median": lambda: make_pipeline(
            SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier(9)
        ),
    }
    res = {k: [] for k in models}
    for m in range(n_masks):
        rng = np.random.default_rng(100 + m)
        X = Xwine.copy()
        X[rng.random(X.shape) < p] = np.nan
        cv = StratifiedKFold(5, shuffle=True, random_state=m)
        for label, make in models.items():
            res[label] += cross_val_score(make(), X, ywine, cv=cv).tolist()
    return [{"missing": p, "method": k, "mean": float(np.mean(v)), "sd": float(np.std(v))} for k, v in res.items()]


t0 = time.time()
miss = pl.DataFrame([r for p in MISSING for r in missing_scores(p)])
print(miss.pivot(on="method", index="missing", values="mean").with_columns(pl.col(pl.Float64).round(3)))
print(f"cell time {time.time() - t0:.0f} s")

In [ ]:
# plot t6_card_missing
fg, axs = plotlib.fig("full", 1.4, ncols=2)
styles = {
    "tree, one-hot": (GREEN, "-", "s"),
    "k-NN, one-hot unscaled": (BLUE, "-", "^"),
    "k-NN, one-hot standardized": (BLUE, "--", "v"),
}
xs = np.arange(len(LEVELS))
for label, (colour, ls, marker) in styles.items():
    sub = card.filter(pl.col("model") == label)
    axs[0].plot(xs, sub["mean"], color=colour, ls=ls, marker=marker, ms=3, label=label)
    axs[0].fill_between(xs, sub["mean"] - sub["sd"], sub["mean"] + sub["sd"], color=colour, alpha=0.1, linewidth=0)
n_cols = card.filter(pl.col("model") == "tree, one-hot")["columns"].to_list()
axs[0].set_xticks(xs, [f"{c}\n({k})" for c, k in zip(LEVELS, n_cols)], fontsize=6.5)
axs[0].set(xlabel="postcode levels (columns)", ylabel="accuracy", ylim=(0.58, 0.77))
axs[0].legend(loc="lower left", fontsize=6.5, ncols=1, handlelength=1.6)
mstyles = {
    "tree, native NaN": (GREEN, "-", "s"),
    "tree, median": (GREEN, "--", "o"),
    "tree, median + flag": (GREEN, ":", "D"),
    "k-NN, median": (BLUE, "-", "^"),
}
for label, (colour, ls, marker) in mstyles.items():
    sub = miss.filter(pl.col("method") == label)
    axs[1].plot(100 * sub["missing"], sub["mean"], color=colour, ls=ls, marker=marker, ms=3, label=label)
    axs[1].fill_between(
        100 * sub["missing"], sub["mean"] - sub["sd"], sub["mean"] + sub["sd"], color=colour, alpha=0.1, linewidth=0
    )
axs[1].set(xlabel="values missing in every column (%)", ylabel="accuracy", ylim=(0.48, 0.80))
axs[1].legend(loc="lower left", fontsize=6.5, ncols=1, handlelength=1.6)
fg.subplots_adjust(wspace=0.35)
display(fg)
plotlib.save(fg, "t6_card_missing")

## T7 — Counting the trees: why the greedy algorithm

Finding the **best** decision tree is NP-complete (Hyafil and Rivest, 1976). Here the claim is made concrete on a tiny
set: all trees of depth at most 2 are enumerated and scored, the number of trees is counted for deeper trees, and the
greedy tree of scikit-learn is compared with the exhaustive optimum on 200 random sets. Timings are typical values of
this machine, not constants.

In [ ]:
# slide: How the Cost of an Algorithm Grows
slide("How the Cost of an Algorithm Grows")
NS7 = np.arange(1, 31)
growth = {
    "n": NS7.astype(float),
    "n log n": NS7 * np.log2(NS7),
    "n^2": NS7.astype(float) ** 2,
    "2^n": 2.0**NS7,
    "n!": np.array([float(math.factorial(int(n))) for n in NS7]),
}
OPS_PER_SECOND = 1e9
SECONDS = {"second": 1.0, "minute": 60.0, "hour": 3600.0, "day": 86400.0, "year": 3.156e7}


def human(t):
    """Seconds as a readable time (the unit nearest to the value)."""
    for unit, size in (
        ("years", 3.156e7),
        ("days", 86400.0),
        ("hours", 3600.0),
        ("minutes", 60.0),
        ("s", 1.0),
        ("ms", 1e-3),
        ("us", 1e-6),
        ("ns", 1e-9),
    ):
        if t >= size:
            return f"{t / size:.2g} {unit}"
    return f"{t / 1e-9:.2g} ns"


rows = []
for n in (10, 20, 30, 50, 100):
    ops = {"n log n": n * math.log2(n), "n^2": float(n**2), "2^n": 2.0**n, "n!": float(math.factorial(n))}
    row = {"n": n}
    for name, v in ops.items():
        t = v / OPS_PER_SECOND
        row[name] = human(t)
    rows.append(row)
print(pl.DataFrame(rows))
print("time at 10^9 operations per second (typical of one core); the age of the universe is about 4e17 s = 1e10 years")

In [ ]:
# plot t7_growth
fg, ax = plotlib.fig("full", 1.5)
for (name, ys), colour, label in zip(
    list(growth.items())[1:], (GREEN, BLUE, ORANGE, RED), ("$n \\log n$", "$n^2$", "$2^n$", "$n!$")
):
    ax.plot(NS7, ys, color=colour, lw=1.4, label=label)
ax.plot(NS7, growth["n"], color=GREY, lw=1.0, label="$n$")
ax.axhline(OPS_PER_SECOND, color="#444444", ls=":", lw=0.8)
ax.axhline(OPS_PER_SECOND * SECONDS["year"], color="#444444", ls=":", lw=0.8)
ax.annotate("1 second", (24, OPS_PER_SECOND), (0, 2), textcoords="offset points", fontsize=7.5, color="#444444")
ax.annotate(
    "1 year", (22, OPS_PER_SECOND * SECONDS["year"]), (0, 2), textcoords="offset points", fontsize=7.5, color="#444444"
)
ax.set(yscale="log", xlabel="problem size $n$", ylabel="operations (log scale)", xlim=(1, 30), ylim=(1, 1e33))
ax.set_yticks([1e0, 1e6, 1e12, 1e18, 1e24, 1e30])
ax.legend(loc="upper left", fontsize=8, ncols=1)
display(fg)
plotlib.save(fg, "t7_growth")

In [ ]:
# slide: The Number of Trees Explodes
slide("The Number of Trees Explodes")


def n_trees(S, h):
    """Trees of depth at most h with S possible questions: a leaf, or a question with two subtrees of depth at most h - 1."""
    t = 1
    for _ in range(h):
        t = 1 + S * t * t
    return t


rows = [{"S": S, **{f"h={h}": f"{n_trees(S, h):.1e}" for h in (1, 2, 3, 4, 5)}} for S in (10, 100, 1000)]
print(pl.DataFrame(rows))
print(
    "S = number of candidate questions (feature, threshold); n_trees(100, 2) =",
    n_trees(100, 2),
    "; for d features and n examples S is about d (n - 1)",
)


def tiny_set(n, seed, kind="xor", d=4, noise=0.15):
    """Tiny set with `noise` of the labels flipped and d - 2 noise columns: kind "xor" is the XOR of two features (the case greedy cannot see), "rule" is the union of two half-lines."""
    rng = np.random.default_rng(seed)
    X = rng.uniform(size=(n, d))
    y = ((X[:, 0] > 0.5) ^ (X[:, 1] > 0.5)) if kind == "xor" else ((X[:, 0] > 0.5) | (X[:, 1] > 0.7))
    return X, np.where(rng.random(n) < noise, 1 - y.astype(int), y.astype(int))


def questions(X):
    """All questions 'x_j <= t' (t between consecutive sorted values) as a boolean matrix (S, n): True = goes left."""
    cols = []
    for j in range(X.shape[1]):
        v = np.sort(np.unique(X[:, j]))
        cols += [X[:, j] <= (a + b) / 2 for a, b in zip(v[:-1], v[1:])]
    return np.array(cols)


def exhaustive_depth2(X, y):
    """Score every tree of depth <= 2 (root question x left subtree x right subtree) and return the smallest number of training errors and the number of trees tried."""
    M = questions(X)
    S = len(M)
    Y1, Y0 = (M & (y == 1)).astype(float), (M & (y == 0)).astype(float)
    best, tried = len(y), 0
    for s in range(S):
        sides = []
        for side in (M[s], ~M[s]):
            n1, n0 = Y1 @ side, Y0 @ side
            t1, t0 = float(np.sum(side & (y == 1))), float(np.sum(side & (y == 0)))
            leaf = min(t1, t0)
            split = np.minimum(n1, n0) + np.minimum(t1 - n1, t0 - n0)
            sides.append(np.r_[leaf, split])  # leaf, or one of the S questions
        total = sides[0][:, None] + sides[1][None, :]  # every (left subtree, right subtree) pair
        best = min(best, float(total.min()))
        tried += total.size
    leaf_only = float(min(np.sum(y == 1), np.sum(y == 0)))
    return min(best, leaf_only), tried + 1


def greedy_depth2(X, y):
    tree = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y)
    return int(np.sum(tree.predict(X) != y))


rows = []
for n in (10, 20, 40, 80, 160, 320):
    X, y = tiny_set(n, seed=n)
    t0 = time.perf_counter()
    best, tried = exhaustive_depth2(X, y)
    t_all = time.perf_counter() - t0
    t0 = time.perf_counter()
    err_greedy = greedy_depth2(X, y)
    t_greedy = time.perf_counter() - t0
    rows.append(
        {
            "n": n,
            "S": len(questions(X)),
            "trees tried": tried,
            "exhaustive (s)": t_all,
            "greedy (s)": t_greedy,
            "best errors": int(best),
            "greedy errors": err_greedy,
        }
    )
timing = pl.DataFrame(rows)
show(timing, 4)
S_big = 4 * 639
per_tree = float(timing["exhaustive (s)"][-1] / timing["trees tried"][-1])
print(
    f"typical time per tree {per_tree:.1e} s; n = 640 (S = {S_big}) would try {n_trees(S_big, 2):.1e} trees, about {per_tree * n_trees(S_big, 2):.0f} s; depth 3 on n = 40: {n_trees(4 * 39, 3):.1e} trees, about {human(per_tree * n_trees(4 * 39, 3))}"
)

In [ ]:
# plot t7_count
fg, axs = plotlib.fig("full", 1.3, ncols=2)
hs = np.arange(1, 6)
for S, colour in ((10, GREEN), (100, BLUE), (1000, RED)):
    axs[0].plot(hs, [n_trees(S, h) for h in hs], color=colour, marker="o", ms=3, label=f"$S$ = {S}")
axs[0].axhline(1e80, color="#444444", ls=":", lw=0.8)
axs[0].annotate("atoms in the universe", (1, 1e80), (2, 3), textcoords="offset points", fontsize=7, color="#444444")
axs[0].set(yscale="log", xlabel="levels $u$", ylabel="trees, up to $u$ levels (log scale)", xticks=hs)
axs[0].legend(loc="upper left", bbox_to_anchor=(0.0, 0.88), fontsize=7)
axs[1].plot(timing["S"], timing["exhaustive (s)"], color=RED, marker="o", ms=3, label="exhaustive, depth 2")
axs[1].plot(timing["S"], timing["greedy (s)"], color=GREEN, marker="s", ms=3, label="greedy, depth 2")
s_proj = np.array([timing["S"][-1], 4 * 639, 4 * 1279])
axs[1].plot(
    s_proj, per_tree * np.array([n_trees(int(s), 2) for s in s_proj]), color=RED, ls="--", lw=1, label="projected"
)
axs[1].set(xscale="log", yscale="log", xlabel="questions $S$ (log scale)", ylabel="seconds (log scale)")
axs[1].legend(loc="upper left", fontsize=7)
fg.subplots_adjust(wspace=0.45)
display(fg)
plotlib.save(fg, "t7_count")

In [ ]:
# slide: Greedy: A Good Tree at a Low Cost
slide("Greedy: A Good Tree at a Low Cost")
results = {}
for kind in ("xor", "rule"):
    gaps, errs, test = [], [], []
    t0 = time.perf_counter()
    for seed in range(200):
        X, y = tiny_set(40, seed=1000 + seed, kind=kind)
        best, _ = exhaustive_depth2(X, y)
        errs.append((best, greedy_depth2(X, y)))
        gaps.append(errs[-1][1] - best)
        if seed < 40:
            Xt, yt = tiny_set(4000, seed=5000 + seed, kind=kind)
            test.append(float(np.mean(DecisionTreeClassifier(max_depth=2, random_state=0).fit(X, y).predict(Xt) != yt)))
    gaps, errs = np.array(gaps, dtype=int), np.array(errs)
    results[kind] = gaps
    print(
        f"{kind}: 200 sets, n = 40, d = 4 (S about 156), {n_trees(156, 2):.1e} trees per set ({time.perf_counter() - t0:.0f} s in all)"
    )
    print(
        f"   greedy finds the optimum on {np.mean(gaps == 0):.3f} of the sets; mean training errors: optimum {errs[:, 0].mean():.2f}, greedy {errs[:, 1].mean():.2f} of 40 ({errs[:, 0].mean() / 40:.3f} and {errs[:, 1].mean() / 40:.3f}); worst gap {gaps.max()}"
    )
    print(f"   greedy tree, test error on 4000 fresh points (40 sets): mean {np.mean(test):.3f}; noise floor 0.150")
    print(
        f"   gap histogram (errors above the optimum: sets): {dict(zip(*(a.tolist() for a in np.unique(gaps, return_counts=True))))}"
    )

In [ ]:
# plot t7_error
fg, axs = plotlib.fig("full", 1.25, ncols=2, sharey=True)
for ax, (kind, title), colour in zip(
    axs, (("rule", "an axis-parallel rule"), ("xor", "XOR of two features")), (GREEN, RED)
):
    vals, counts = np.unique(results[kind], return_counts=True)
    ax.bar(vals, counts, color=colour, width=0.7)
    ax.set_xticks(np.arange(0, vals.max() + 1, 2))
    ax.set_title(title, fontsize=7.5, pad=3)
    ax.set(xlabel="errors above the optimum")
    ax.text(
        0.97,
        0.9,
        f"optimal on {np.mean(results[kind] == 0):.0%}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=7.5,
    )
axs[0].set(ylabel="data sets (of 200)")
display(fg)
plotlib.save(fg, "t7_error")

## Summary

* A tree, a linear model and k-NN all end as a **partition of the feature space into cells**; the shape of the cells (half-plane, box, Voronoi polygon) is what differs. With enough cells any region is reached, the price is data and dimension.
* A tree is a sequence of questions, grown **greedily** by the impurity gain (Gini or entropy); a numeric feature is split
  at a midpoint found by a sorted scan with cumulative counts.
* A category of $c$ levels offers $2^{c-1} - 1$ questions; encode it knowing what the next model does with the numbers (a tree: little, k-NN: a distance).
* Finding the best tree is NP-complete: the number of trees explodes with the depth, so the greedy tree is the working answer; it can miss a good question that needs a mediocre one first (XOR).
* Trees overfit by construction: limit the depth or the leaf size, or prune by cost-complexity, **on validation data**.
* Regression trees predict leaf means (a staircase) and cannot extrapolate.
* Rules are readable; the *importances* and the first split are **unstable** and biased: use permutation importance and,
  next class, ensembles of trees.